# Connectivity 1 — Point-process GLM functional connectivity (REM)

| | |
|---|---|
| **Original** | `Liam_work/tests/graphical_analysis/AD_PointProcess_GLM_Connectivity.ipynb` |
| **Status** | Side project (Feb 2026). Produces the GLM pickle used by the other connectivity notebooks. |

Fits an L1-regularised point-process GLM (Pillow 2008 / Truccolo 2005; raised-cosine history and coupling
bases, 1 ms bins) per neuron, within each brain region, during REM, for all 20 animals. Coupling weights
give a directed functional-connectivity matrix per (animal, region). Compares connectivity statistics
AD vs WT, overall and by cell-type pair (RSU→RSU, RSU→FS, FS→RSU, FS→FS).

**Input.** Raw single-unit + sleep-score files under `/media/HlabShare/appps1_cortical_communication_samb/`.
**Output.** `data/glm_results/glm_connectivity_results_20animals_REM.pkl` (~45 MB, not in git).
Liam's original wrote to `tests/glm_results/`, but the downstream notebooks read from
`tests/graphical_analysis/glm_results/` (the file was moved by hand); the repo copies all use
`data/glm_results/`.

**Runtime.** Long — ~5–10 min per animal.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# Point Process GLM: Neural Connectivity Analysis
## AD vs WT Functional Connectivity via Spike Train Coupling

**Method**: L1-regularized Point Process GLM with raised cosine basis functions  
**Reference**: Pillow et al. (2008) Nature; Truccolo et al. (2005) J Neurophysiol  

**Approach**:  
For each neuron, model its probability of spiking as a function of:
1. Its own spike history (refractory/bursting dynamics)
2. Coupling from other simultaneously recorded neurons

The coupling weights reveal directed functional connectivity.  
We then compare connectivity patterns between AD (APP/PS1) and WT mice.

**Brain state: REM sleep** — chosen because:
- Desynchronized cortical state (no UP/DOWN oscillation confound)
- No behavioral variability (unlike wake)
- AD specifically disrupts REM-related circuits (theta-gamma coupling, cholinergic function)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from collections import Counter, defaultdict
import warnings
import os
import time
import pickle

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 8)
print("Imports loaded.")

## Configuration


In [ ]:
DATA_DIR = '/media/HlabShare/appps1_cortical_communication_samb/appps1/data/'
# Repo copy: write results into the project's data/ folder (never into Liam_work).
RESULTS_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/glm_results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

# 20 animals — matches ISI analysis (all_animal_isi_df.pkl)
ANIMAL_INFO = {
    'BTH90':  'WT',  'BTH95':  'WT',  'BTH100': 'WT',
    'CAF42':  'WT',  'CAF69':  'WT',  'CAF77':  'WT',
    'CAF81':  'WT',  'CAF82':  'WT',  'CAF88':  'WT',
    'CAF104': 'WT',  'ZBR106': 'WT',
    'CAF71':  'AD',  'CAF73':  'AD',  'CAF74':  'AD',  'CAF75': 'AD',
    'ZBR113': 'AD',  'ZBR114': 'AD',  'ZBR117': 'AD',
    'ZBR118': 'AD',  'ZBR119': 'AD',
}

# GLM parameters
BIN_SIZE_MS = 1
N_HISTORY_BASIS = 8
N_COUPLING_BASIS = 8
HISTORY_WINDOW_MS = 100   # self-history filter: 100ms (captures refractory + bursting)
COUPLING_WINDOW_MS = 50   # coupling filter: 50ms (captures mono/polysynaptic)
REM_WINDOW_MIN = 10       # use 10 minutes of REM per animal
REG_C = 'ebic'            # Extended BIC (Chen & Chen 2008) — penalizes model space size
EBIC_GAMMA = 1.0          # EBIC gamma: 0=standard BIC, 0.5=moderate, 1=conservative (Chen & Chen 2008)

# Sleep state mapping (hengenlab convention):
# 1 = Active Wake (AW), 2 = NREM, 3 = REM, 4 = Quiet Wake (QW)
TARGET_STATE = 3  # REM — desynchronized state, no UP/DOWN confound

# Parallelization
N_JOBS = 15

FS = 25000.0
SLEEP_BIN_S = 4

print(f"Config: {BIN_SIZE_MS}ms bins, {N_COUPLING_BASIS} coupling basis over {COUPLING_WINDOW_MS}ms")
print(f"Using {REM_WINDOW_MIN} min of REM (state={TARGET_STATE}), L1 C={REG_C}")
print(f"EBIC gamma={EBIC_GAMMA} (Chen & Chen 2008, Biometrika) — conservative, consistent selection")
print(f"No population rate covariate needed — REM lacks UP/DOWN state shared drive")
print(f"Parallel workers: {N_JOBS} / {os.cpu_count()} cores")
print(f"Animals: {sum(1 for v in ANIMAL_INFO.values() if v=='WT')} WT, "
      f"{sum(1 for v in ANIMAL_INFO.values() if v=='AD')} AD")

## Raised Cosine Basis Functions

Log-spaced raised cosine basis functions (Pillow et al. 2008) efficiently parameterize
temporal filters with high resolution near zero lag (where monosynaptic effects occur)
and coarser resolution at longer lags.


In [ ]:
def raised_cosine_basis(n_basis, n_bins, a=1.5):
    """
    Log-spaced raised cosine basis functions.
    
    Parameters
    ----------
    n_basis : int        Number of basis functions
    n_bins : int         Number of time bins the basis spans
    a : float            Log-spacing offset (larger = more uniform spacing)
    
    Returns
    -------
    basis : ndarray (n_bins, n_basis)
    """
    centers = np.linspace(np.log(1 + a), np.log(n_bins + a), n_basis)
    width = (centers[1] - centers[0]) if n_basis > 1 else 2.0
    
    t = np.arange(1, n_bins + 1)
    log_t = np.log(t + a)
    
    basis = np.zeros((n_bins, n_basis))
    for i, c in enumerate(centers):
        x = (log_t - c) / width
        mask = np.abs(x) <= 1
        basis[mask, i] = (np.cos(x[mask] * np.pi) + 1) / 2
    
    # Normalize
    for i in range(n_basis):
        s = basis[:, i].sum()
        if s > 0:
            basis[:, i] /= s
    
    return basis

# Create and visualize
history_basis = raised_cosine_basis(N_HISTORY_BASIS, HISTORY_WINDOW_MS)
coupling_basis = raised_cosine_basis(N_COUPLING_BASIS, COUPLING_WINDOW_MS)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i in range(N_HISTORY_BASIS):
    axes[0].plot(np.arange(1, HISTORY_WINDOW_MS+1), history_basis[:, i], label=f'b{i}')
axes[0].set_xlabel('Lag (ms)')
axes[0].set_ylabel('Weight')
axes[0].set_title(f'History Basis ({N_HISTORY_BASIS} functions, {HISTORY_WINDOW_MS}ms)')
axes[0].legend(fontsize=7, ncol=2)

for i in range(N_COUPLING_BASIS):
    axes[1].plot(np.arange(1, COUPLING_WINDOW_MS+1), coupling_basis[:, i], label=f'b{i}')
axes[1].set_xlabel('Lag (ms)')
axes[1].set_title(f'Coupling Basis ({N_COUPLING_BASIS} functions, {COUPLING_WINDOW_MS}ms)')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()
print(f"History basis: {history_basis.shape}, Coupling basis: {coupling_basis.shape}")


## Data Loading & Spike Binning


In [ ]:
def find_recordings(animal_name, data_dir=DATA_DIR):
    """Find all singleunit + sleep file pairs for an animal."""
    animal_dir = os.path.join(data_dir, animal_name)
    recordings = []
    for root, dirs, files in os.walk(animal_dir):
        for f in sorted(files):
            if f.endswith('_singleunits.npy'):
                sleep_f = f.replace('_singleunits.npy', '_sleep.npy')
                sleep_path = os.path.join(root, sleep_f)
                if os.path.exists(sleep_path):
                    recordings.append({
                        'neurons_path': os.path.join(root, f),
                        'sleep_path': sleep_path,
                        'name': f.replace('_singleunits.npy', '')
                    })
    return recordings


def _get_neuron_attr(neuron, *attrs, default='unknown'):
    """Get first available attribute from a neuron object.
    Old neurons (CAF69-82) use .animal, new ones use .animal_name, etc."""
    for a in attrs:
        val = getattr(neuron, a, None)
        if val is not None:
            return val
    return default


def get_state_intervals(sleep_states, target_state=TARGET_STATE):
    """Convert sleep state bins to sample-level intervals.
    
    Sleep state mapping (hengenlab convention):
        1 = Active Wake, 2 = NREM, 3 = REM, 4 = Quiet Wake
    """
    intervals = []
    in_state = False
    start = None
    for i, s in enumerate(sleep_states):
        if s == target_state and not in_state:
            start = int(i * SLEEP_BIN_S * FS)
            in_state = True
        elif s != target_state and in_state:
            end = int(i * SLEEP_BIN_S * FS)
            intervals.append((start, end))
            in_state = False
    if in_state:
        intervals.append((start, int(len(sleep_states) * SLEEP_BIN_S * FS)))
    return intervals


def bin_spikes_in_state(neurons, intervals, max_ms=None, animal_name=None):
    """
    Bin spike trains into 1ms binary arrays during target state intervals.
    
    Parameters
    ----------
    neurons : array of Neuron objects
    intervals : list of (start, end) sample-level tuples
    max_ms : int or None, max milliseconds to use
    animal_name : str, fallback animal name if neuron object lacks it
    
    Returns
    -------
    binned : ndarray (n_neurons, n_bins), int8
    info : list of dicts per neuron
    """
    bin_samples = int(BIN_SIZE_MS * FS / 1000)
    
    total_samples = sum(e - s for s, e in intervals)
    n_bins = total_samples // bin_samples
    if max_ms is not None:
        n_bins = min(n_bins, int(max_ms / BIN_SIZE_MS))
    
    binned = np.zeros((len(neurons), n_bins), dtype=np.int8)
    
    cumulative = 0
    for start, end in intervals:
        seg_bins = (end - start) // bin_samples
        if cumulative + seg_bins > n_bins:
            seg_bins = n_bins - cumulative
            end = start + seg_bins * bin_samples
        
        for i, neuron in enumerate(neurons):
            st = neuron.spike_time
            mask = (st >= start) & (st < end)
            bin_idx = (st[mask] - start) // bin_samples
            bin_idx = bin_idx[bin_idx < seg_bins]
            binned[i, cumulative + bin_idx] = 1
        
        cumulative += seg_bins
        if cumulative >= n_bins:
            break
    
    binned = binned[:, :cumulative]
    
    # Look up genotype from ANIMAL_INFO since old neurons lack .genotype
    geno = ANIMAL_INFO.get(animal_name, '?') if animal_name else '?'
    
    info = []
    for i, neuron in enumerate(neurons):
        info.append({
            'region': getattr(neuron, 'region', 'unknown'),
            'cell_type': getattr(neuron, 'cell_type', 'unknown'),
            'animal': _get_neuron_attr(neuron, 'animal_name', 'animal'),
            'genotype': getattr(neuron, 'genotype', geno),
            'n_spikes': int(binned[i].sum()),
            'firing_rate_hz': float(binned[i].sum()) / (cumulative * BIN_SIZE_MS / 1000)
        })
    
    return binned, info

STATE_NAMES = {1: 'Active Wake', 2: 'NREM', 3: 'REM', 4: 'Quiet Wake'}
print(f"Target state: {TARGET_STATE} ({STATE_NAMES.get(TARGET_STATE, '?')})")

# Verify recordings exist
for animal in sorted(ANIMAL_INFO.keys()):
    recs = find_recordings(animal)
    print(f"  {animal} ({ANIMAL_INFO[animal]}): {len(recs)} recordings")

## Design Matrix Construction

For target neuron $j$, the GLM predicts spiking probability as:

$$\log \frac{P(\text{spike}_j(t))}{1 - P(\text{spike}_j(t))} = \beta_0 + \sum_k \alpha_k \cdot h_k(t) + \sum_{i \neq j} \sum_k \beta_{ik} \cdot c_{ik}(t)$$

Where:
- $h_k(t) = \sum_{\tau=1}^{K} b_k^{\text{hist}}(\tau) \cdot y_j(t-\tau)$ captures self-history
- $c_{ik}(t) = \sum_{\tau=1}^{K} b_k^{\text{coup}}(\tau) \cdot y_i(t-\tau)$ captures coupling from neuron $i$

The convolution is **strictly causal** (minimum lag = 1 bin = 1ms).


In [ ]:
def build_design_matrix(binned, target_idx, hist_basis, coup_basis,
                       neuron_subset=None, keep_idx=None):
    """
    Build design matrix for one target neuron.
    
    Features (in order):
      1. Self-history basis convolutions (n_hb columns)
      2. Coupling basis convolutions per source (n_sources * n_cb columns)
    
    No population rate covariate needed for REM — the desynchronized state
    lacks UP/DOWN oscillations, so shared drive doesn't inflate coupling.
    
    If keep_idx is provided, only those rows are stored (saves ~20x memory).
    The convolutions still run on the full signal for correctness.
    
    Returns
    -------
    X : (n_keep, n_features) float64
    y : (n_keep,) int8
    source_indices : list of int
    """
    n_neurons, T = binned.shape
    K_h = hist_basis.shape[0]
    K_c = coup_basis.shape[0]
    max_lag = max(K_h, K_c)
    n_valid = T - max_lag
    n_hb = hist_basis.shape[1]
    n_cb = coup_basis.shape[1]
    
    if neuron_subset is None:
        neuron_subset = [i for i in range(n_neurons) if i != target_idx]
    else:
        neuron_subset = [i for i in neuron_subset if i != target_idx]
    
    n_features = n_hb + len(neuron_subset) * n_cb
    n_rows = len(keep_idx) if keep_idx is not None else n_valid
    X = np.empty((n_rows, n_features), dtype=np.float64)
    
    col = 0
    # 1. History features
    target_f = binned[target_idx].astype(np.float64)
    for b in range(n_hb):
        conv = np.convolve(target_f, hist_basis[:, b])
        full_col = conv[max_lag - 1 : max_lag - 1 + n_valid]
        X[:, col] = full_col[keep_idx] if keep_idx is not None else full_col
        col += 1
    
    # 2. Coupling features
    for src in neuron_subset:
        src_f = binned[src].astype(np.float64)
        for b in range(n_cb):
            conv = np.convolve(src_f, coup_basis[:, b])
            full_col = conv[max_lag - 1 : max_lag - 1 + n_valid]
            X[:, col] = full_col[keep_idx] if keep_idx is not None else full_col
            col += 1
    
    y_full = binned[target_idx, max_lag : max_lag + n_valid]
    y = y_full[keep_idx] if keep_idx is not None else y_full
    return X, y, neuron_subset

print("Design matrix builder ready.")
print(f"Features per target: {N_HISTORY_BASIS} history + N_sources x {N_COUPLING_BASIS} coupling")

## GLM Fitting & Connectivity Extraction

For each target neuron:
1. Build design matrix (history + all within-region coupling)
2. Fit L1-regularized logistic regression
3. Extract coupling weights per source neuron
4. Coupling strength = L2 norm of basis coefficients (signed by dominant component)

**Within-region analysis**: neurons on the same probe are most likely to have direct connections.


In [ ]:
def _fit_single_neuron(binned, tgt_global, neuron_info, region_idx,
                       hist_basis, coup_basis, C, max_iter,
                       neg_ratio=10):
    """Fit GLM for one target neuron. Designed to run in parallel.
    
    neg_ratio : int
        Keep this many non-spike bins per spike bin. Only affects intercept,
        not coupling weights (Prentice & Pyke 1979).
    """
    info = neuron_info[tgt_global]
    n_cb = coup_basis.shape[1]
    n_hb = hist_basis.shape[1]
    ti = region_idx.index(tgt_global)
    n_reg = len(region_idx)
    
    result = {
        'ti': ti,
        'tgt_global': tgt_global,
        'conn_row': np.zeros(n_reg),
        'filters': {},
        'status': 'skipped',
        'n_connections': 0,
    }
    
    if info['n_spikes'] < 100:
        result['status'] = 'too_few_spikes'
        return result
    
    # Compute subsample indices BEFORE building design matrix
    max_lag = max(hist_basis.shape[0], coup_basis.shape[0])
    T = binned.shape[1]
    n_valid = T - max_lag
    y_full = binned[tgt_global, max_lag : max_lag + n_valid]
    
    n_pos = int(y_full.sum())
    if n_pos < 50:
        result['status'] = 'too_few_target_spikes'
        return result
    
    pos_idx = np.where(y_full == 1)[0]
    neg_idx = np.where(y_full == 0)[0]
    n_neg_keep = min(neg_ratio * n_pos, len(neg_idx))
    
    rng = np.random.RandomState(42 + tgt_global)
    neg_sample = rng.choice(neg_idx, size=n_neg_keep, replace=False)
    keep_idx = np.sort(np.concatenate([pos_idx, neg_sample]))
    
    # Build subsampled design matrix
    X, y, sources = build_design_matrix(
        binned, tgt_global, hist_basis, coup_basis,
        neuron_subset=region_idx, keep_idx=keep_idx
    )
    
    # Standardize
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    
    # Select C via EBIC (Chen & Chen 2008, Biometrika)
    # EBIC = -2*logL + k*log(n) + 2*gamma*k*log(p)
    # Standard BIC (gamma=0) is too permissive with large n — selects
    # too many spurious connections. The extra 2*gamma*k*log(p) term
    # penalizes model complexity proportional to the search space.
    if C in ('bic', 'ebic'):
        C_candidates = np.logspace(-5, -1, 15)  # C=0.00001 to 0.1
        best_ebic = np.inf
        best_C = C_candidates[0]
        model = None
        n_samples = len(y)
        p_features = X_s.shape[1]  # total candidate features
        for C_try in C_candidates:
            m = LogisticRegression(
                penalty='l1', C=C_try, solver='liblinear',
                max_iter=max_iter, class_weight='balanced',
                random_state=42, tol=1e-3
            )
            m.fit(X_s, y)
            log_proba = m.predict_log_proba(X_s)
            log_lik = np.sum(log_proba[np.arange(n_samples), y.astype(int)])
            k = np.sum(m.coef_ != 0) + 1  # +1 for intercept
            # EBIC: BIC + model space penalty
            ebic = (-2 * log_lik
                    + k * np.log(n_samples)
                    + 2 * EBIC_GAMMA * k * np.log(p_features))
            if ebic < best_ebic:
                best_ebic = ebic
                best_C = C_try
                model = m
        C = best_C
    else:
        model = LogisticRegression(
            penalty='l1', C=C, solver='liblinear',
            max_iter=max_iter, class_weight='balanced',
            random_state=42, tol=1e-3
        )
        model.fit(X_s, y)
    
    # Extract coupling weights (skip history columns)
    coefs = model.coef_[0]
    coup_coefs = coefs[n_hb:]
    
    conn_row = np.zeros(n_reg)
    filters = {}
    
    for si_local, src_global in enumerate(sources):
        si_in_region = region_idx.index(src_global)
        w = coup_coefs[si_local * n_cb : (si_local + 1) * n_cb]
        
        strength = np.linalg.norm(w)
        if strength > 0:
            sign = np.sign(w[np.argmax(np.abs(w))])
            conn_row[si_in_region] = strength * sign
        
        filt = coup_basis @ w
        filters[(src_global, tgt_global)] = filt
    
    result['conn_row'] = conn_row
    result['filters'] = filters
    result['status'] = 'fitted'
    result['n_connections'] = int(np.sum(np.abs(conn_row) > 1e-6))
    result['n_samples'] = len(y)
    result['n_pos'] = n_pos
    result['C_used'] = C
    return result


def fit_region_glm(binned, neuron_info, hist_basis, coup_basis,
                   region, C=REG_C, max_iter=500, n_jobs=N_JOBS):
    """Fit GLM for all neurons within a brain region IN PARALLEL."""
    region_idx = [i for i, info in enumerate(neuron_info)
                  if info['region'].lower().replace(' ', '') == region.lower().replace(' ', '')]
    
    if len(region_idx) < 3:
        print(f"    {region}: only {len(region_idx)} neurons, skipping")
        return None, region_idx, {}
    
    n_reg = len(region_idx)
    n_features = N_HISTORY_BASIS + (n_reg - 1) * N_COUPLING_BASIS
    print(f"    {region}: {n_reg} neurons, {n_features} features/neuron, "
          f"{n_jobs} workers, C={C}...", flush=True)
    
    t0 = time.time()
    results = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(_fit_single_neuron)(
            binned, tgt_global, neuron_info, region_idx,
            hist_basis, coup_basis, C, max_iter
        )
        for tgt_global in region_idx
    )
    elapsed = time.time() - t0
    
    # Assemble connectivity matrix
    conn = np.zeros((n_reg, n_reg))
    coupling_filters = {}
    n_fitted = 0
    n_skipped = 0
    
    for res in results:
        if res['status'] == 'fitted':
            conn[:, res['ti']] = res['conn_row']
            coupling_filters.update(res['filters'])
            n_fitted += 1
        else:
            n_skipped += 1
    
    n_connections = np.sum(np.abs(conn) > 1e-6)
    n_possible = n_reg * (n_reg - 1)
    density = 100 * n_connections / max(n_possible, 1)
    print(f"    >> {region} done in {elapsed:.0f}s | "
          f"{n_fitted} fitted, {n_skipped} skipped | "
          f"{n_connections}/{n_possible} connections ({density:.1f}%)")
    
    return conn, region_idx, coupling_filters


def analyze_animal(animal_name, state_min=REM_WINDOW_MIN, C=REG_C):
    """Full pipeline for one animal."""
    recs = find_recordings(animal_name)
    if not recs:
        print(f"  No recordings found for {animal_name}")
        return None
    
    rec = recs[0]
    t_load = time.time()
    neurons = np.load(rec['neurons_path'], allow_pickle=True)
    sleep = np.load(rec['sleep_path'], allow_pickle=True)
    print(f"  Loaded {rec['name']} in {time.time()-t_load:.1f}s: "
          f"{len(neurons)} neurons, {len(sleep)} sleep bins")
    
    unique_states, state_counts = np.unique(sleep, return_counts=True)
    target_count = state_counts[unique_states == TARGET_STATE]
    state_name = STATE_NAMES.get(TARGET_STATE, f'state {TARGET_STATE}')
    if len(target_count) == 0 or target_count[0] == 0:
        print(f"  WARNING: No {state_name} (state={TARGET_STATE}) found in {rec['name']}")
        return None
    
    state_min_available = float(target_count[0]) * SLEEP_BIN_S / 60
    print(f"  {state_name}={state_min_available:.1f}min available, "
          f"using {min(state_min, state_min_available):.1f}min")
    
    if state_min_available < 1.0:
        print(f"  WARNING: Less than 1 min of {state_name}, skipping {animal_name}")
        return None
    
    intervals = get_state_intervals(sleep)
    t_bin = time.time()
    max_ms = state_min * 60 * 1000
    binned, neuron_info = bin_spikes_in_state(neurons, intervals, max_ms=max_ms,
                                               animal_name=animal_name)
    
    regions = sorted(set(info['region'] for info in neuron_info))
    region_counts = Counter(info['region'] for info in neuron_info)
    print(f"  Binned in {time.time()-t_bin:.1f}s: {binned.shape[1]/1000/60:.1f}min @ 1ms, "
          f"{binned.nbytes/1e6:.0f}MB | Regions: {dict(region_counts)}")
    
    hist_basis = raised_cosine_basis(N_HISTORY_BASIS, HISTORY_WINDOW_MS)
    coup_basis = raised_cosine_basis(N_COUPLING_BASIS, COUPLING_WINDOW_MS)
    
    results = {
        'animal': animal_name,
        'genotype': ANIMAL_INFO.get(animal_name, '?'),
        'n_neurons': len(neurons),
        'neuron_info': neuron_info,
        'regions': {},
    }
    
    for ri, region in enumerate(regions, 1):
        print(f"  [{ri}/{len(regions)}] Fitting {region}...")
        conn, idx, filters = fit_region_glm(
            binned, neuron_info, hist_basis, coup_basis,
            region=region, C=C
        )
        if conn is not None:
            results['regions'][region] = {
                'connectivity': conn,
                'neuron_indices': idx,
                'coupling_filters': filters,
            }
    
    return results

print("Analysis pipeline ready.")
print(f"Sleep state: {STATE_NAMES.get(TARGET_STATE, '?')} (state {TARGET_STATE}) | Parallel: {N_JOBS} workers")
print(f"Regularization: EBIC gamma={EBIC_GAMMA} (Chen & Chen 2008)")
print(f"C range: 1e-5 to 0.1 (tighter than standard BIC)")
print(f"Expected density: 5-15% (vs 60-90% with standard BIC)")
print(f"Negative subsampling: 10:1 ratio | Memory-efficient design matrix")

## Run Analysis: All Animals

Fits within-region GLM connectivity for all 20 animals (11 WT, 9 AD) during REM sleep.  
Animals with less than 1 minute of REM will be skipped.  
Estimated time: ~5-10 min per animal with BIC selection + 15 parallel workers.

In [ ]:
all_results = {}
animals = sorted(ANIMAL_INFO.keys())
total_t0 = time.time()

state_name = STATE_NAMES.get(TARGET_STATE, f'state{TARGET_STATE}')
print(f"Running GLM connectivity on {state_name} data for {len(animals)} animals\n")

for ai, animal in enumerate(animals, 1):
    print(f"\n{'='*70}")
    print(f"[{ai}/{len(animals)}] {animal} ({ANIMAL_INFO[animal]})")
    print(f"{'='*70}")
    
    animal_t0 = time.time()
    result = analyze_animal(animal)
    animal_elapsed = time.time() - animal_t0
    
    if result is not None:
        all_results[animal] = result
        n_regions = len(result['regions'])
        total_conn = sum(
            np.sum(np.abs(rd['connectivity']) > 1e-6)
            for rd in result['regions'].values()
        )
        print(f"  >> {animal} done in {animal_elapsed:.1f}s | "
              f"{n_regions} regions, {total_conn} total connections")
    else:
        print(f"  >> {animal} FAILED or SKIPPED (insufficient {state_name})")
    
    # Progress estimate
    elapsed_total = time.time() - total_t0
    avg_per_animal = elapsed_total / ai
    remaining = (len(animals) - ai) * avg_per_animal
    print(f"  >> Overall: {elapsed_total/60:.1f}min elapsed, "
          f"~{remaining/60:.1f}min remaining")

total_elapsed = time.time() - total_t0
print(f"\n{'='*70}")
print(f"ALL DONE: {len(all_results)}/{len(animals)} animals in {total_elapsed/60:.1f} min")
print(f"  WT: {sum(1 for r in all_results.values() if r['genotype']=='WT')}")
print(f"  AD: {sum(1 for r in all_results.values() if r['genotype']=='AD')}")
if len(all_results) < len(animals):
    skipped = set(animals) - set(all_results.keys())
    print(f"  Skipped ({len(skipped)}): {sorted(skipped)}")
print(f"{'='*70}")

# Save results — separate file from NREM results
save_path = os.path.join(RESULTS_DIR, 'glm_connectivity_results_20animals_REM.pkl')
save_data = {}
for animal, res in all_results.items():
    save_data[animal] = {
        'animal': res['animal'],
        'genotype': res['genotype'],
        'n_neurons': res['n_neurons'],
        'neuron_info': res['neuron_info'],
        'regions': {
            reg: {
                'connectivity': rdata['connectivity'],
                'neuron_indices': rdata['neuron_indices'],
                'coupling_filters': rdata['coupling_filters'],
            }
            for reg, rdata in res['regions'].items()
        }
    }
with open(save_path, 'wb') as f:
    pickle.dump(save_data, f)
print(f"\nResults saved to {save_path}")
print(f"(NREM results preserved at glm_connectivity_results_20animals.pkl)")

## Visualize Connectivity

1. **Connectivity matrices** per region (heatmaps)
2. **Coupling filter examples** (temporal profiles of strongest connections)


In [ ]:
def plot_connectivity(result, top_n_filters=6):
    """Plot connectivity matrices and top coupling filters for one animal."""
    animal = result['animal']
    geno = result['genotype']
    regions = result['regions']
    neuron_info = result['neuron_info']
    
    n_regions = len(regions)
    if n_regions == 0:
        print(f"{animal}: No regions to plot.")
        return
    
    # --- Connectivity matrices ---
    fig, axes = plt.subplots(1, n_regions, figsize=(5 * n_regions, 4))
    if n_regions == 1:
        axes = [axes]
    fig.suptitle(f'{animal} ({geno}) - Connectivity Matrices', fontsize=14)
    
    for ax, (region, rdata) in zip(axes, sorted(regions.items())):
        conn = rdata['connectivity']
        idx = rdata['neuron_indices']
        
        vmax = np.percentile(np.abs(conn[conn != 0]), 95) if np.any(conn != 0) else 1
        im = ax.imshow(conn, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
        
        types = [neuron_info[i]['cell_type'] for i in idx]
        ax.set_xticks(range(len(idx)))
        ax.set_xticklabels(types, fontsize=5, rotation=90)
        ax.set_yticks(range(len(idx)))
        ax.set_yticklabels(types, fontsize=5)
        
        n_conn = np.sum(np.abs(conn) > 1e-6)
        density = n_conn / max(conn.size - len(idx), 1) * 100
        ax.set_title(f'{region}\n{len(idx)} neurons, {density:.0f}% dense')
        ax.set_xlabel('Target')
        ax.set_ylabel('Source')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.tight_layout()
    plt.show()
    
    # --- Top coupling filters ---
    all_filters = {}
    for region, rdata in regions.items():
        for (src, tgt), filt in rdata.get('coupling_filters', {}).items():
            strength = np.abs(filt).max()
            if strength > 0:
                all_filters[(src, tgt)] = {
                    'filter': filt, 'strength': strength, 'region': region,
                    'src_info': neuron_info[src], 'tgt_info': neuron_info[tgt],
                }
    
    if not all_filters:
        return
    
    top = sorted(all_filters.items(), key=lambda x: -x[1]['strength'])[:top_n_filters]
    n_plots = min(len(top), top_n_filters)
    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.flat
    fig.suptitle(f'{animal} ({geno}) - Strongest Coupling Filters', fontsize=14)
    
    for ax, ((src, tgt), info) in zip(axes, top):
        t_ms = np.arange(1, len(info['filter']) + 1)
        ax.plot(t_ms, info['filter'], 'b-', linewidth=2)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        color = 'red' if info['filter'].max() > abs(info['filter'].min()) else 'blue'
        ax.fill_between(t_ms, 0, info['filter'], alpha=0.3, color=color)
        ax.set_xlabel('Lag (ms)')
        ax.set_ylabel('Coupling weight')
        ax.set_title(f"{info['region']}: {info['src_info']['cell_type']}→"
                     f"{info['tgt_info']['cell_type']} ({info['strength']:.3f})")
    
    for ax in list(axes)[n_plots:]:
        ax.set_visible(False)
    
    plt.tight_layout()
    plt.show()

# Plot for each animal
for animal, result in all_results.items():
    plot_connectivity(result)

In [ ]:
def extract_connectivity_stats(results):
    """Extract summary statistics from each animal's connectivity."""
    stats_list = []
    
    for animal, res in results.items():
        for region, rdata in res['regions'].items():
            conn = rdata['connectivity']
            idx = rdata['neuron_indices']
            info = res['neuron_info']
            n = conn.shape[0]
            
            if n < 3:
                continue
            
            # Remove diagonal
            mask = ~np.eye(n, dtype=bool)
            weights = conn[mask]
            nonzero = weights[np.abs(weights) > 1e-6]
            
            stats_list.append({
                'animal': animal,
                'genotype': res['genotype'],
                'region': region,
                'n_neurons': n,
                'n_connections': len(nonzero),
                'n_possible': n * (n - 1),
                'density': len(nonzero) / (n * (n - 1)) if n > 1 else 0,
                'mean_strength': np.mean(np.abs(nonzero)) if len(nonzero) > 0 else 0,
                'median_strength': np.median(np.abs(nonzero)) if len(nonzero) > 0 else 0,
                'n_excitatory': np.sum(nonzero > 0),
                'n_inhibitory': np.sum(nonzero < 0),
                'ei_ratio': np.sum(nonzero > 0) / max(np.sum(nonzero < 0), 1),
                'mean_exc': np.mean(nonzero[nonzero > 0]) if np.any(nonzero > 0) else 0,
                'mean_inh': np.mean(np.abs(nonzero[nonzero < 0])) if np.any(nonzero < 0) else 0,
            })
    
    return stats_list


def compare_ad_wt(results):
    """Statistical comparison of AD vs WT connectivity."""
    stats_list = extract_connectivity_stats(results)
    
    if not stats_list:
        print("No data to compare.")
        return
    
    regions = sorted(set(s['region'] for s in stats_list))
    
    # --- Figure 1: Density and Strength by genotype ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('AD vs WT Functional Connectivity', fontsize=14, fontweight='bold')
    
    # 1. Connection density by region
    ax = axes[0, 0]
    for gi, geno in enumerate(['WT', 'AD']):
        for ri, region in enumerate(regions):
            vals = [s['density'] for s in stats_list
                    if s['genotype'] == geno and s['region'] == region]
            if vals:
                x = ri + gi * 0.35 - 0.175
                ax.bar(x, np.mean(vals), 0.3,
                       yerr=np.std(vals) if len(vals) > 1 else 0,
                       color=['#2196F3', '#F44336'][gi], alpha=0.7,
                       label=geno if ri == 0 else '')
                for v in vals:
                    ax.scatter(x, v, color='black', s=15, zorder=5)
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(regions, rotation=45)
    ax.set_ylabel('Connection Density')
    ax.set_title('Connection Density by Region')
    ax.legend()
    
    # 2. Mean connection strength
    ax = axes[0, 1]
    for gi, geno in enumerate(['WT', 'AD']):
        for ri, region in enumerate(regions):
            vals = [s['mean_strength'] for s in stats_list
                    if s['genotype'] == geno and s['region'] == region]
            if vals:
                x = ri + gi * 0.35 - 0.175
                ax.bar(x, np.mean(vals), 0.3,
                       yerr=np.std(vals) if len(vals) > 1 else 0,
                       color=['#2196F3', '#F44336'][gi], alpha=0.7,
                       label=geno if ri == 0 else '')
                for v in vals:
                    ax.scatter(x, v, color='black', s=15, zorder=5)
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(regions, rotation=45)
    ax.set_ylabel('Mean |Coupling Weight|')
    ax.set_title('Connection Strength by Region')
    ax.legend()
    
    # 3. E/I ratio
    ax = axes[1, 0]
    for gi, geno in enumerate(['WT', 'AD']):
        for ri, region in enumerate(regions):
            vals = [s['ei_ratio'] for s in stats_list
                    if s['genotype'] == geno and s['region'] == region]
            if vals:
                x = ri + gi * 0.35 - 0.175
                ax.bar(x, np.mean(vals), 0.3,
                       yerr=np.std(vals) if len(vals) > 1 else 0,
                       color=['#2196F3', '#F44336'][gi], alpha=0.7,
                       label=geno if ri == 0 else '')
                for v in vals:
                    ax.scatter(x, v, color='black', s=15, zorder=5)
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(regions, rotation=45)
    ax.set_ylabel('E/I Ratio')
    ax.set_title('Excitatory/Inhibitory Ratio')
    ax.axhline(1, color='gray', linestyle='--', alpha=0.5)
    ax.legend()
    
    # 4. Overall density: AD vs WT
    ax = axes[1, 1]
    wt_dens = [s['density'] for s in stats_list if s['genotype'] == 'WT']
    ad_dens = [s['density'] for s in stats_list if s['genotype'] == 'AD']
    
    parts = ax.violinplot([wt_dens, ad_dens] if wt_dens and ad_dens else [[0], [0]],
                          positions=[0, 1], showmeans=True, showmedians=True)
    if wt_dens and ad_dens:
        stat, pval = stats.mannwhitneyu(wt_dens, ad_dens, alternative='two-sided')
        ax.set_title(f'Overall Density\nMann-Whitney p={pval:.4f}')
    else:
        ax.set_title('Overall Density')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['WT', 'AD'])
    ax.set_ylabel('Connection Density')
    
    plt.tight_layout()
    plt.show()
    
    # --- Statistical tests per region ---
    print("\n" + "="*60)
    print("STATISTICAL TESTS (Mann-Whitney U, per region)")
    print("="*60)
    for region in regions:
        wt = [s['density'] for s in stats_list
              if s['genotype'] == 'WT' and s['region'] == region]
        ad = [s['density'] for s in stats_list
              if s['genotype'] == 'AD' and s['region'] == region]
        
        if len(wt) >= 2 and len(ad) >= 2:
            stat, pval = stats.mannwhitneyu(wt, ad, alternative='two-sided')
            print(f"  {region}: WT density={np.mean(wt):.3f}±{np.std(wt):.3f}, "
                  f"AD density={np.mean(ad):.3f}±{np.std(ad):.3f}, p={pval:.4f}")
        else:
            print(f"  {region}: WT n={len(wt)}, AD n={len(ad)} (too few for test)")

compare_ad_wt(all_results)


## Cell-Type Specific Connectivity

Break down connections by cell type pairs: RSU→RSU, RSU→FS, FS→RSU, FS→FS


In [ ]:
def celltype_connectivity(results):
    """Analyze connectivity by cell-type pairs."""
    pair_types = ['RSU→RSU', 'RSU→FS', 'FS→RSU', 'FS→FS']
    
    data = defaultdict(lambda: defaultdict(list))  # geno -> pair_type -> [strengths]
    
    for animal, res in results.items():
        geno = res['genotype']
        info = res['neuron_info']
        
        for region, rdata in res['regions'].items():
            conn = rdata['connectivity']
            idx = rdata['neuron_indices']
            n = len(idx)
            
            for si in range(n):
                for ti in range(n):
                    if si == ti:
                        continue
                    w = conn[si, ti]
                    if abs(w) < 1e-6:
                        continue
                    
                    src_type = info[idx[si]]['cell_type']
                    tgt_type = info[idx[ti]]['cell_type']
                    pair = f"{src_type}→{tgt_type}"
                    
                    if pair in pair_types:
                        data[geno][pair].append(abs(w))
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Cell-Type Specific Connectivity: AD vs WT', fontsize=14)
    
    # Bar chart of mean strength
    ax = axes[0]
    x = np.arange(len(pair_types))
    width = 0.35
    
    for gi, geno in enumerate(['WT', 'AD']):
        means = [np.mean(data[geno][pt]) if data[geno][pt] else 0 for pt in pair_types]
        sems = [np.std(data[geno][pt])/np.sqrt(len(data[geno][pt]))
                if len(data[geno][pt]) > 1 else 0 for pt in pair_types]
        counts = [len(data[geno][pt]) for pt in pair_types]
        
        bars = ax.bar(x + gi*width - width/2, means, width,
                      yerr=sems, color=['#2196F3', '#F44336'][gi],
                      alpha=0.7, label=f'{geno}')
        
        for xi, c in zip(x + gi*width - width/2, counts):
            ax.text(xi, 0.001, f'n={c}', ha='center', va='bottom', fontsize=7)
    
    ax.set_xticks(x)
    ax.set_xticklabels(pair_types)
    ax.set_ylabel('Mean |Coupling Weight|')
    ax.set_title('Connection Strength by Cell-Type Pair')
    ax.legend()
    
    # Connection counts
    ax = axes[1]
    for gi, geno in enumerate(['WT', 'AD']):
        counts = [len(data[geno][pt]) for pt in pair_types]
        ax.bar(x + gi*width - width/2, counts, width,
               color=['#2196F3', '#F44336'][gi], alpha=0.7, label=geno)
    
    ax.set_xticks(x)
    ax.set_xticklabels(pair_types)
    ax.set_ylabel('Number of Connections')
    ax.set_title('Connection Count by Cell-Type Pair')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Stats
    print("\nCell-type pair statistics:")
    for pt in pair_types:
        wt_vals = data['WT'][pt]
        ad_vals = data['AD'][pt]
        if len(wt_vals) >= 2 and len(ad_vals) >= 2:
            stat, pval = stats.mannwhitneyu(wt_vals, ad_vals, alternative='two-sided')
            print(f"  {pt}: WT mean={np.mean(wt_vals):.4f} (n={len(wt_vals)}), "
                  f"AD mean={np.mean(ad_vals):.4f} (n={len(ad_vals)}), p={pval:.4f}")
        else:
            print(f"  {pt}: WT n={len(wt_vals)}, AD n={len(ad_vals)}")

celltype_connectivity(all_results)


## Summary

### Method
- **Point Process GLM** with L1 regularization (Pillow et al. 2008)
- Raised cosine basis functions: 8 history (100ms), 8 coupling (50ms)
- Within-region analysis during **REM sleep** (desynchronized state)
- Directed functional connectivity: coupling weight i→j
- No population rate covariate needed — REM lacks UP/DOWN shared drive

### Why REM?
- NREM UP/DOWN oscillations inflate pairwise coupling (77% density with NREM)
- REM is desynchronized — coupling estimates reflect true circuit interactions
- AD disrupts REM-specific circuits (theta-gamma, cholinergic system)
- REM showed best genotype classification in spike metrics analysis

### Metrics Compared (AD vs WT)
1. **Connection density**: fraction of neuron pairs with non-zero coupling
2. **Connection strength**: magnitude of coupling weights
3. **E/I ratio**: excitatory vs inhibitory connection balance
4. **Cell-type specificity**: RSU↔RSU, RSU↔FS, FS↔RSU, FS↔FS

### Key Question
Does Alzheimer's pathology (APP/PS1) alter the functional connectivity
structure of neural circuits during REM sleep?